<a href="https://colab.research.google.com/github/aparnap-seven/ICT_DSA_CaseStudy/blob/main/casestudy_data_acquisition.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

 # Data Acquisition and Analysis from SpaceX Data
 Ingest SpaceX launch and rocket data from APIs, clean and merge the data, and store it in a
local SQLite3 database. Use SQL queries to analyze the launch data.


## Step 1: Load SpaceX Launch Data from API
● Use https://api.spacexdata.com/v4/launches
● Extract relevant columns: name, date_utc, success, details, rocket
● Convert date_utc to datetime and extract the year

In [ ]:
import requests
import pandas as pd
url = 'https://api.spacexdata.com/v4/launches'
response = requests.get(url)
launch_data = response.json()

df_launch = pd.DataFrame(launch_data)


In [ ]:
print(df_launch.columns)

Index(['fairings', 'links', 'static_fire_date_utc', 'static_fire_date_unix',
       'net', 'window', 'rocket', 'success', 'failures', 'details', 'crew',
       'ships', 'capsules', 'payloads', 'launchpad', 'flight_number', 'name',
       'date_utc', 'date_unix', 'date_local', 'date_precision', 'upcoming',
       'cores', 'auto_update', 'tbd', 'launch_library_id', 'id'],
      dtype='object')


In [8]:
#To select the required columns - name, date_utc, success, details, rocket
df_launch = df_launch[['name', 'date_utc', 'success', 'details', 'rocket']]

#To convert date_utc to date time
df_launch['date_utc'] = pd.to_datetime(df_launch['date_utc'])

#To extract year
df_launch['year'] = df_launch['date_utc'].dt.year

df_launch.head(5)

,name,date_utc,success,details,rocket,year
0,FalconSat,2006-03-24 22:30:00+00:00,False,Engine failure at 33 seconds and loss of vehicle,5e9d0d95eda69955f709d1eb,2006
1,DemoSat,2007-03-21 01:10:00+00:00,False,Successful first stage burn and transition to ...,5e9d0d95eda69955f709d1eb,2007
2,Trailblazer,2008-08-03 03:34:00+00:00,False,Residual stage 1 thrust led to collision betwe...,5e9d0d95eda69955f709d1eb,2008
3,RatSat,2008-09-28 23:15:00+00:00,True,Ratsat was carried to orbit on the first succe...,5e9d0d95eda69955f709d1eb,2008
4,RazakSat,2009-07-13 03:35:00+00:00,True,None,5e9d0d95eda69955f709d1eb,2009


## Step 2: Load Rocket Metadata
● Use https://api.spacexdata.com/v4/rockets
● Extract id, name, type, active, and stages

In [13]:
url_rocket = 'https://api.spacexdata.com/v4/rockets'
response = requests.get(url_rocket)
rocket_data = response.json()
df_rocket = pd.DataFrame(rocket_data)

#To extract id, name, type, active, and stages
df_rocket = df_rocket[['id', 'name', 'type', 'active', 'stages']]

df_rocket.head(5)

,id,name,type,active,stages
0,5e9d0d95eda69955f709d1eb,Falcon 1,rocket,False,2
1,5e9d0d95eda69973a809d1ec,Falcon 9,rocket,True,2
2,5e9d0d95eda69974db09d1ed,Falcon Heavy,rocket,True,2
3,5e9d0d96eda699382d09d1ee,Starship,rocket,False,2


## Step 3: Merge Launch and Rocket Data
● Join the two DataFrames on rocket ID using

In [15]:
#To merge launch data with rocket data
df_merged = pd.merge(df_launch, df_rocket,
                     left_on='rocket', right_on='id')
df_merged.head(5)

,name_x,date_utc,success,details,rocket,year,id,name_y,type,active,stages
0,FalconSat,2006-03-24 22:30:00+00:00,False,Engine failure at 33 seconds and loss of vehicle,5e9d0d95eda69955f709d1eb,2006,5e9d0d95eda69955f709d1eb,Falcon 1,rocket,False,2
1,DemoSat,2007-03-21 01:10:00+00:00,False,Successful first stage burn and transition to ...,5e9d0d95eda69955f709d1eb,2007,5e9d0d95eda69955f709d1eb,Falcon 1,rocket,False,2
2,Trailblazer,2008-08-03 03:34:00+00:00,False,Residual stage 1 thrust led to collision betwe...,5e9d0d95eda69955f709d1eb,2008,5e9d0d95eda69955f709d1eb,Falcon 1,rocket,False,2
3,RatSat,2008-09-28 23:15:00+00:00,True,Ratsat was carried to orbit on the first succe...,5e9d0d95eda69955f709d1eb,2008,5e9d0d95eda69955f709d1eb,Falcon 1,rocket,False,2
4,RazakSat,2009-07-13 03:35:00+00:00,True,None,5e9d0d95eda69955f709d1eb,2009,5e9d0d95eda69955f709d1eb,Falcon 1,rocket,False,2


## Step 4: Add Simulated Country Information
● Add a new column country and randomly assign one of these values:
['USA', 'Russia', 'India', 'China', 'France']


In [16]:
#To add simulated country column randomly
import random
countries =['USA', 'Russia', 'India', 'China', 'France']
df_merged['country'] = [random.choice(countries) for _ in range(len(df_merged))]
df_merged.head(5)

,name_x,date_utc,success,details,rocket,year,id,name_y,type,active,stages,country
0,FalconSat,2006-03-24 22:30:00+00:00,False,Engine failure at 33 seconds and loss of vehicle,5e9d0d95eda69955f709d1eb,2006,5e9d0d95eda69955f709d1eb,Falcon 1,rocket,False,2,France
1,DemoSat,2007-03-21 01:10:00+00:00,False,Successful first stage burn and transition to ...,5e9d0d95eda69955f709d1eb,2007,5e9d0d95eda69955f709d1eb,Falcon 1,rocket,False,2,China
2,Trailblazer,2008-08-03 03:34:00+00:00,False,Residual stage 1 thrust led to collision betwe...,5e9d0d95eda69955f709d1eb,2008,5e9d0d95eda69955f709d1eb,Falcon 1,rocket,False,2,France
3,RatSat,2008-09-28 23:15:00+00:00,True,Ratsat was carried to orbit on the first succe...,5e9d0d95eda69955f709d1eb,2008,5e9d0d95eda69955f709d1eb,Falcon 1,rocket,False,2,France
4,RazakSat,2009-07-13 03:35:00+00:00,True,None,5e9d0d95eda69955f709d1eb,2009,5e9d0d95eda69955f709d1eb,Falcon 1,rocket,False,2,Russia


## Step 5: Store Merged Data in SQLite3
● Use sqlite3 to create a connection and save the merged DataFrame as a table named
launches
● Table should contain all merged columns including country

In [17]:
#To store in SQL
import sqlite3
conn = sqlite3.connect('spacexdata.db')
df_merged.to_sql('launches', conn, if_exists='replace', index=False)
print("Data saved successfully!")

Data saved successfully!


In [18]:
#To show table contain all merged values
query = "SELECT * FROM launches LIMIT 5;"
pd.read_sql(query, conn)

,name_x,date_utc,success,details,rocket,year,id,name_y,type,active,stages,country
0,FalconSat,2006-03-24 22:30:00+00:00,0,Engine failure at 33 seconds and loss of vehicle,5e9d0d95eda69955f709d1eb,2006,5e9d0d95eda69955f709d1eb,Falcon 1,rocket,0,2,France
1,DemoSat,2007-03-21 01:10:00+00:00,0,Successful first stage burn and transition to ...,5e9d0d95eda69955f709d1eb,2007,5e9d0d95eda69955f709d1eb,Falcon 1,rocket,0,2,China
2,Trailblazer,2008-08-03 03:34:00+00:00,0,Residual stage 1 thrust led to collision betwe...,5e9d0d95eda69955f709d1eb,2008,5e9d0d95eda69955f709d1eb,Falcon 1,rocket,0,2,France
3,RatSat,2008-09-28 23:15:00+00:00,1,Ratsat was carried to orbit on the first succe...,5e9d0d95eda69955f709d1eb,2008,5e9d0d95eda69955f709d1eb,Falcon 1,rocket,0,2,France
4,RazakSat,2009-07-13 03:35:00+00:00,1,None,5e9d0d95eda69955f709d1eb,2009,5e9d0d95eda69955f709d1eb,Falcon 1,rocket,0,2,Russia


## Step 6: Run SQL Queries on the Data to analyze
1. Launches by Country
2. Which year had the highest number of launches?
3. Top 5 Missions by Launch Count

In [20]:
#To show launches by country
q1 = """
SELECT country, COUNT(*) AS total_launch
FROM launches
GROUP BY country;
"""
pd.read_sql(q1, conn)

,country,total_launch
0,China,41
1,France,39
2,India,42
3,Russia,40
4,USA,43


In [21]:
#To show which year has the highest no.of launches
q2 = """
SELECT year, COUNT(*) as total
FROM launches
GROUP BY year
ORDER BY total DESC
LIMIT 1;
"""

pd.read_sql(q2, conn)

,year,total
0,2022,62


In [23]:
#To show Top 5 missions by Launch count
q3 = """
SELECT name_x, COUNT(*) AS launch_count
FROM launches
GROUP BY name_x
ORDER BY launch_count DESC
LIMIT 5;
"""

pd.read_sql(q3, conn)

,name_x,launch_count
0,ispace Mission 1 & Rashid,1
1,ZUMA,1
2,WorldView Legion 1 & 2,1
3,Viasat-3 & Arcturus,1
4,USSF-44,1


In [24]:
conn.close()